In [2]:
import os
import requests
from bs4 import BeautifulSoup
import pandas as pd
from google.colab import files
from IPython.display import display

# ==========================================
# 1. AUTOMATIC FILE DOWNLOAD
# ==========================================
HTML_URL = "https://raw.githubusercontent.com/noodlejacknetwork/porsche-scraper-avelino-ccs7-sir-ibs/refs/heads/main/porsche.html"
FILE_NAME = "porsche.html"

if not os.path.exists(FILE_NAME):
    print("Downloading porsche.html from cloud storage...")
    response = requests.get(HTML_URL)
    with open(FILE_NAME, "wb") as f:
        f.write(response.content)
    print("Download complete!\n")
else:
    print("Using existing porsche.html file.\n")

print("--- Processing HTML File ---")

# ==========================================
# 2. PARSE THE DATA
# ==========================================
with open(FILE_NAME, "r", encoding="utf-8") as file:
    html_content = file.read()

soup = BeautifulSoup(html_content, 'html.parser')
car_cards = soup.find_all('div', attrs={"data-test": "model-overview-range_model-card"})
print(f"Total cars successfully found: {len(car_cards)}\n")

cars_data = []
for card in car_cards:
    name_elem = card.find(attrs={"data-test": "model-card-header_title"})
    name = name_elem.text.strip() if name_elem else "Unknown Model"

    list_items = card.find_all('li')
    accel, power, top_speed = None, None, None

    if len(list_items) >= 3:
        accel = list_items[0].find('div').text.strip()
        power = list_items[1].find('div').text.strip()
        top_speed = list_items[2].find('div').text.strip()

    cars_data.append({
        "Model": name,
        "0-100 km/h": accel,
        "Power": power,
        "Top Speed": top_speed
    })

# Convert to initial DataFrame
df = pd.DataFrame(cars_data)

# ==========================================
# 3. CLEAN AND CONVERT THE COLUMNS
# ==========================================
print("Cleaning columns and transforming text to numerical data types...")

# Clean Acceleration and Top Speed
df['0-100 km/h'] = df['0-100 km/h'].str.replace(' s', '', regex=False).astype(float)
df['Top Speed'] = df['Top Speed'].str.replace(' km/h', '', regex=False).astype(float)

# Split Power (kW and PS)
power_split = df['Power'].str.split(' / ', expand=True)

# Clean kW: Remove ' kW', remove commas, convert to float
df['Power (kW)'] = power_split[0].str.replace(' kW', '', regex=False).str.replace(',', '', regex=False).astype(float)

# Clean PS: Remove ' PS', remove commas, convert to float
df['Power (PS)'] = power_split[1].str.replace(' PS', '', regex=False).str.replace(',', '', regex=False).astype(float)

# Drop old mixed column and rename others cleanly
df = df.drop(columns=['Power'])
df = df.rename(columns={
    '0-100 km/h': '0-100 km/h (s)',
    'Top Speed': 'Top Speed (km/h)'
})

print("\n--- Final Cleaned Dataset Info ---")
print(df.info())

# ==========================================
# 4. EXPORT TO CSV
# ==========================================
csv_filename = "porsche_data_cleaned.csv"
df.to_csv(csv_filename, index=False)
print(f"\n--- Data successfully saved to {csv_filename} ---")

print("\n--- Previewing Final Data ---")
display(df.head())

# Download the CSV
files.download(csv_filename)

Using existing porsche.html file.

--- Processing HTML File ---
Total cars successfully found: 88

Cleaning columns and transforming text to numerical data types...

--- Final Cleaned Dataset Info ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 88 entries, 0 to 87
Data columns (total 5 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   Model             88 non-null     object 
 1   0-100 km/h (s)    88 non-null     float64
 2   Top Speed (km/h)  88 non-null     float64
 3   Power (kW)        88 non-null     float64
 4   Power (PS)        88 non-null     float64
dtypes: float64(4), object(1)
memory usage: 3.6+ KB
None

--- Data successfully saved to porsche_data_cleaned.csv ---

--- Previewing Final Data ---


,Model,0-100 km/h (s),Top Speed (km/h),Power (kW),Power (PS)
0,718 Cayman,4.9,275.0,220.0,300.0
1,718 Cayman Style Edition,4.9,275.0,220.0,300.0
2,718 Cayman S,4.4,285.0,257.0,350.0
3,718 Cayman GTS 4.0,4.0,288.0,294.0,400.0
4,718 Boxster,4.9,275.0,220.0,300.0


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>